# Baseline ResNet-50 dengan 5-Fold Cross-Validation di Google Colab

Notebook ini melatih **baseline ResNet-50 tanpa segmentation guidance** menggunakan citra parenkim dari `000_dataset_v2`. Data dibaca dari arsip `.tar.gz` di Google Drive, disalin ke penyimpanan lokal Colab, lalu diperiksa sebelum training dimulai. Hasil training dan salinan konfigurasi JSON disimpan kembali ke Google Drive.

Sebelum mulai, push kode terbaru ke branch yang dikonfigurasi, lalu pilih **Runtime > Change runtime type > GPU**. Jalankan semua cell secara berurutan dari atas ke bawah. Progress bar `tqdm` akan terlihat ketika arsip disalin, data diperiksa, serta ketika training dan validation berjalan.

> Catatan: satu `EXPERIMENT_ID` hanya boleh dipakai untuk satu training baru. Ganti nilainya jika ingin membuat eksperimen lain.

## Persiapan arsip sebelum membuka Google Colab

Jalankan perintah berikut dari direktori utama repository pada komputer lokal. Arsip hanya berisi metadata cross-validation, citra parenkim, dan ground-truth mask. **Probability map tidak disertakan dan tidak digunakan oleh baseline ResNet-50.** Mask hanya diperlukan untuk tahap XAI opsional; mask bukan input model ketika training.

```bash
tar -czf a0d90f9e-3dd4-4de0-98af-12858696f613_cv_resnet50_parenchyma.tar.gz \
  000_dataset_v2/_segmentation_dataset/004_classification_cv_5fold_seed42.csv \
  000_dataset_v2/_lidc/007_segmentation_dataset_npy/ct_parenchyma \
  000_dataset_v2/_lndb/007_segmentation_dataset_npy/ct_parenchyma \
  000_dataset_v2/_lidc/007_segmentation_dataset_npy/mask \
  000_dataset_v2/_lndb/007_segmentation_dataset_npy/mask
```

Setelah selesai, upload file tersebut ke:

`MyDrive/mask-guided-lung-nodule-xai/a0d90f9e-3dd4-4de0-98af-12858696f613_cv_resnet50_parenchyma.tar.gz`

Struktur direktori di dalam arsip harus tetap diawali dengan `000_dataset_v2`. Jangan memasukkan folder probability map.

## 1. Periksa GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU tidak tersedia. Aktifkan GPU pada pengaturan runtime Colab."
    )

print(f"PyTorch version: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Hubungkan Google Drive

Arsip dataset dibaca dari Google Drive. Semua hasil eksperimen juga disimpan di sana agar tidak hilang ketika runtime Colab berhenti.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Atur konfigurasi eksperimen

Ini adalah cell konfigurasi utama. Ubah repository, lokasi arsip, nama eksperimen, model, training, optimizer, atau DataLoader di sini sebelum menjalankan cell berikutnya. `CT_PATH_COLUMN` sengaja memakai `ct_parenchyma_path`.

In [ ]:
from pathlib import Path

# Repository
REPOSITORY_URL = (
    "https://github.com/FillipusAditya/"
    "mask-guided-lung-nodule-xai.git"
)
REPOSITORY_BRANCH = "refactor/002-segmentation"
PROJECT_ROOT = Path("/content/mask-guided-lung-nodule-xai")

# Eksperimen
EXPERIMENT_ID = "a0d90f9e-3dd4-4de0-98af-12858696f613"
EXPERIMENT_COMPONENT = "classification/cv_resnet50"

# Dataset
DRIVE_PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mask-guided-lung-nodule-xai"
)
ARCHIVE_NAME = f"{EXPERIMENT_ID}_cv_resnet50_parenchyma.tar.gz"
DRIVE_ARCHIVE_PATH = DRIVE_PROJECT_ROOT / ARCHIVE_NAME
LOCAL_ARCHIVE_PATH = Path("/content") / ARCHIVE_NAME
EXTRACTION_ROOT = Path("/content/classification_training_data")
DATASET_ROOT = EXTRACTION_ROOT / "000_dataset_v2/_segmentation_dataset"
COPY_ARCHIVE_TO_LOCAL = True
FORCE_EXTRACT = False

# Output
DRIVE_OUTPUT_ROOT = DRIVE_PROJECT_ROOT / "experiment_results"
DRIVE_OUTPUT_DIR = (
    DRIVE_OUTPUT_ROOT / EXPERIMENT_ID / EXPERIMENT_COMPONENT
)

# Input data
METADATA_FILENAME = "004_classification_cv_5fold_seed42.csv"
CT_PATH_COLUMN = "ct_parenchyma_path"
INPUT_HEIGHT = 224
INPUT_WIDTH = 224
NUM_FOLDS = 5
CLASS_TO_IDX = {"benign": 0, "malignant": 1}
NORMALIZATION_MEAN = [0.485, 0.456, 0.406]
NORMALIZATION_STD = [0.229, 0.224, 0.225]

# Model ResNet-50
PRETRAINED_WEIGHTS = "DEFAULT"
CLASSIFIER_DROPOUT = 0.3

# Training
NUM_EPOCHS = 100
BATCH_SIZE = 32  # Kurangi jika GPU kehabisan memori.
LEARNING_RATE = 1e-3
SEED = 42
TRANSFORM_SEED = 42
CLASSIFICATION_THRESHOLD = 0.5
DEVICE = "auto"

# Optimizer SGD
MOMENTUM = 0.9
WEIGHT_DECAY = 1e-4
NESTEROV = False

# Early stopping
EARLY_STOPPING_PATIENCE = 20
EARLY_STOPPING_MIN_DELTA = 0.0

# DataLoader
NUM_WORKERS = 2
PERSISTENT_WORKERS = True
PREFETCH_FACTOR = 2
PIN_MEMORY = True

# Pengujian opsional setelah training
RUN_TEST_AFTER_TRAINING = False
MAX_TEST_SAMPLES = None  # Gunakan 8 untuk smoke test yang cepat.
TEST_BATCH_SIZE = 2
TEST_NUM_WORKERS = 0

print(f"Dataset: {DATASET_ROOT}")
print(f"Output: {DRIVE_OUTPUT_DIR}")

## 4. Clone atau perbarui repository

In [ ]:
import subprocess

if (PROJECT_ROOT / ".git").is_dir():
    print("Memperbarui repository yang sudah ada...")
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "fetch", "origin"],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "checkout", REPOSITORY_BRANCH],
        check=True,
    )
    subprocess.run(
        [
            "git", "-C", str(PROJECT_ROOT), "pull", "--ff-only",
            "origin", REPOSITORY_BRANCH,
        ],
        check=True,
    )
else:
    print("Mengunduh repository...")
    subprocess.run(
        [
            "git", "clone", "--branch", REPOSITORY_BRANCH,
            "--single-branch", REPOSITORY_URL, str(PROJECT_ROOT),
        ],
        check=True,
    )

training_script = PROJECT_ROOT / "003_classification/cv_resnet50/train.py"
if not training_script.is_file():
    raise FileNotFoundError(f"Script training tidak ditemukan: {training_script}")

print(f"Repository siap: {PROJECT_ROOT}")

## 5. Instal package yang dibutuhkan

Instalasi PyTorch bawaan Colab dipertahankan karena sudah sesuai dengan CUDA pada runtime aktif.

In [ ]:
import sys

packages = [
    "albumentations>=2.0,<3.0",
    "opencv-python-headless",
    "pandas",
    "matplotlib",
    "scikit-learn",
    "tqdm",
    "zennit==0.5.1",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *packages],
    check=True,
)

print("Semua dependency siap.")

## 6. Salin dan ekstrak dataset

Arsip yang diharapkan adalah `a0d90f9e-3dd4-4de0-98af-12858696f613_cv_resnet50_parenchyma.tar.gz` yang berisi direktori teratas `000_dataset_v2`. Penyalinan dan ekstraksi menampilkan progress bar `tqdm`. Set `FORCE_EXTRACT=True` hanya jika hasil ekstraksi lokal perlu dibuat ulang.

In [ ]:
import shutil
import tarfile

from tqdm.auto import tqdm


def copy_file_with_progress(source, destination, chunk_size=8 * 1024 * 1024):
    """Salin satu file sambil menampilkan progress dalam byte."""

    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = destination.with_suffix(destination.suffix + ".part")
    total_bytes = source.stat().st_size

    with source.open("rb") as input_file:
        with temporary_path.open("wb") as output_file:
            with tqdm(
                total=total_bytes,
                desc="Menyalin arsip",
                unit="B",
                unit_scale=True,
            ) as progress_bar:
                while True:
                    chunk = input_file.read(chunk_size)
                    if not chunk:
                        break
                    output_file.write(chunk)
                    progress_bar.update(len(chunk))

    temporary_path.replace(destination)


if not DRIVE_ARCHIVE_PATH.is_file():
    raise FileNotFoundError(f"Arsip dataset tidak ditemukan: {DRIVE_ARCHIVE_PATH}")

if COPY_ARCHIVE_TO_LOCAL:
    archive_path = LOCAL_ARCHIVE_PATH
    source_size = DRIVE_ARCHIVE_PATH.stat().st_size
    local_copy_is_current = (
        archive_path.is_file() and archive_path.stat().st_size == source_size
    )

    if local_copy_is_current:
        print(f"Menggunakan arsip lokal: {archive_path}")
    else:
        copy_file_with_progress(DRIVE_ARCHIVE_PATH, archive_path)
else:
    archive_path = DRIVE_ARCHIVE_PATH

extraction_marker = EXTRACTION_ROOT / ".extraction_complete"

if FORCE_EXTRACT and EXTRACTION_ROOT.exists():
    shutil.rmtree(EXTRACTION_ROOT)

if extraction_marker.is_file():
    print(f"Menggunakan dataset lokal: {EXTRACTION_ROOT}")
else:
    EXTRACTION_ROOT.mkdir(parents=True, exist_ok=True)

    with tarfile.open(archive_path, mode="r:gz") as archive:
        members = archive.getmembers()
        for member in tqdm(members, desc="Mengekstrak dataset", unit="file"):
            archive.extract(member, path=EXTRACTION_ROOT, filter="data")

    extraction_marker.touch()

print(f"Dataset siap: {DATASET_ROOT}")

## 7. Validasi dataset `000_dataset_v2`

Cell ini memeriksa kolom metadata CV, pembagian lima fold, kelas, dan seluruh file parenkim yang akan dipakai untuk training. Mask juga diperiksa karena dibutuhkan jika pengujian XAI opsional dijalankan.

In [ ]:
import csv

metadata_path = DATASET_ROOT / METADATA_FILENAME
required_directories = [
    EXTRACTION_ROOT / "000_dataset_v2/_lidc/007_segmentation_dataset_npy/ct_parenchyma",
    EXTRACTION_ROOT / "000_dataset_v2/_lndb/007_segmentation_dataset_npy/ct_parenchyma",
    EXTRACTION_ROOT / "000_dataset_v2/_lidc/007_segmentation_dataset_npy/mask",
    EXTRACTION_ROOT / "000_dataset_v2/_lndb/007_segmentation_dataset_npy/mask",
]
required_columns = {
    "dataset", "patient_id", "filename", CT_PATH_COLUMN,
    "mask_path", "label", "cv_group_id", "cv_nodule_id",
    "cv_role", "cv_fold",
}

if not metadata_path.is_file():
    raise FileNotFoundError(f"Metadata tidak ditemukan: {metadata_path}")

for directory in required_directories:
    if not directory.is_dir():
        raise FileNotFoundError(f"Direktori tidak ditemukan: {directory}")

with metadata_path.open("r", encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)
    column_names = set(reader.fieldnames or [])
    rows = list(reader)

missing_columns = required_columns - column_names
if missing_columns:
    raise ValueError(f"Kolom metadata tidak lengkap: {sorted(missing_columns)}")
if not rows:
    raise ValueError("Metadata tidak boleh kosong.")

missing_files = []
for row in tqdm(rows, desc="Memeriksa path dataset", unit="sampel"):
    ct_path = DATASET_ROOT / row[CT_PATH_COLUMN]
    mask_path = DATASET_ROOT / row["mask_path"]

    if not ct_path.is_file():
        missing_files.append(ct_path)
    if not mask_path.is_file():
        missing_files.append(mask_path)

if missing_files:
    examples = "\n".join(str(path) for path in missing_files[:5])
    raise FileNotFoundError(
        f"Ada {len(missing_files)} file yang tidak ditemukan.\n{examples}"
    )

development_folds = {
    int(row["cv_fold"])
    for row in rows
    if row["cv_role"].strip().lower() == "development"
}
if development_folds != set(range(NUM_FOLDS)):
    raise ValueError(f"Fold tidak sesuai: {sorted(development_folds)}")

label_counts = {}
for row in rows:
    label = row["label"].strip().lower()
    label_counts[label] = label_counts.get(label, 0) + 1

print(f"Jumlah sampel: {len(rows):,}")
print(f"Distribusi kelas: {label_counts}")
print(f"Fold development: {sorted(development_folds)}")
print("Semua file parenkim dan mask tersedia.")

## 8. Buat dan simpan konfigurasi JSON

Dictionary di bawah dibuat langsung dari cell konfigurasi utama. JSON disimpan ke repository sementara agar dibaca oleh `train.py`, dan satu salinan persisten disimpan ke Google Drive. Ketika training dimulai, `train.py` juga menyalin JSON yang sama ke direktori hasil eksperimen sebagai `cv_resnet50.json`.

In [ ]:
import json

if NUM_FOLDS != 5:
    raise ValueError("Metadata ini harus menggunakan NUM_FOLDS = 5.")
if BATCH_SIZE < 1 or NUM_EPOCHS < 1 or LEARNING_RATE <= 0:
    raise ValueError("Batch size, epoch, dan learning rate harus positif.")
if NUM_WORKERS < 0:
    raise ValueError("NUM_WORKERS tidak boleh negatif.")
if not 0.0 <= CLASSIFICATION_THRESHOLD <= 1.0:
    raise ValueError("CLASSIFICATION_THRESHOLD harus berada antara 0 dan 1.")
if DEVICE not in {"auto", "cpu", "cuda"}:
    raise ValueError("DEVICE harus 'auto', 'cpu', atau 'cuda'.")
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("DEVICE='cuda', tetapi GPU tidak tersedia.")
config = {
    "experiment": {
        "id": EXPERIMENT_ID,
        "component": EXPERIMENT_COMPONENT,
    },
    "output": {
        "root_directory": str(DRIVE_OUTPUT_ROOT),
        "config_snapshot_filename": "cv_resnet50.json",
    },
    "data": {
        "dataset_root": str(DATASET_ROOT),
        "metadata_path": str(metadata_path),
        "ct_path_column": CT_PATH_COLUMN,
        "input_height": INPUT_HEIGHT,
        "input_width": INPUT_WIDTH,
        "class_to_idx": CLASS_TO_IDX,
        "normalization_mean": NORMALIZATION_MEAN,
        "normalization_std": NORMALIZATION_STD,
    },
    "cross_validation": {
        "num_folds": NUM_FOLDS,
        "development_role": "development",
        "holdout_role": "holdout_test",
        "holdout_fold": -1,
        "group_column": "cv_group_id",
        "nodule_column": "cv_nodule_id",
        "fold_column": "cv_fold",
        "role_column": "cv_role",
    },
    "model": {
        "architecture": "ResNet50",
        "pretrained_weights": PRETRAINED_WEIGHTS,
        "training_strategy": "full_fine_tuning",
        "trainable_component": "entire_model",
        "classifier_dropout": CLASSIFIER_DROPOUT,
    },
    "training": {
        "num_epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "seed": SEED,
        "transform_seed": TRANSFORM_SEED,
        "classification_threshold": CLASSIFICATION_THRESHOLD,
        "device": DEVICE,
    },
    "optimizer": {
        "name": "SGD",
        "momentum": MOMENTUM,
        "weight_decay": WEIGHT_DECAY,
        "nesterov": NESTEROV,
    },
    "dataloader": {
        "num_workers": NUM_WORKERS,
        "persistent_workers": PERSISTENT_WORKERS,
        "prefetch_factor": PREFETCH_FACTOR,
        "pin_memory": PIN_MEMORY,
        "train_shuffle": True,
        "val_shuffle": False,
        "train_drop_last": False,
        "val_drop_last": False,
    },
    "early_stopping": {
        "enabled": True,
        "monitor": "val_loss",
        "mode": "min",
        "patience": EARLY_STOPPING_PATIENCE,
        "min_delta": EARLY_STOPPING_MIN_DELTA,
        "verbose": True,
        "restore_best_weights": True,
    },
    "checkpoint": {"save_latest": True},
}


def save_json(data, output_path):
    """Simpan dictionary sebagai file JSON yang mudah dibaca."""

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(data, file, indent=4)
        file.write("\n")


config_path = PROJECT_ROOT / "003_classification/configs/cv_resnet50.json"
drive_config_path = (
    DRIVE_OUTPUT_ROOT / "saved_configs" / EXPERIMENT_ID / "cv_resnet50.json"
)
save_json(config, config_path)
save_json(config, drive_config_path)

print(json.dumps(config, indent=4))
print(f"Config repository: {config_path}")
print(f"Config Google Drive: {drive_config_path}")

## 9. Jalankan preflight check

Preflight menggunakan konfigurasi dan transform yang sama dengan training. Cell ini mengambil satu sampel validation, membangun ResNet-50, dan memastikan bentuk input serta output model benar. Bobot pretrained ImageNet akan diunduh saat model pertama kali dibuat.

In [ ]:
import importlib

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

importlib.invalidate_caches()
module_name = "003_classification.cv_resnet50.train"
if module_name in sys.modules:
    train_module = importlib.reload(sys.modules[module_name])
else:
    train_module = importlib.import_module(module_name)

train_loader, val_loader, _, _ = (
    train_module.engine.build_fold_dataloaders(fold=0)
)
sample_image, sample_label = val_loader.dataset[0]
model = train_module.engine.base_train.build_model(
    num_classes=len(CLASS_TO_IDX)
)
model.eval()

with torch.no_grad():
    sample_output = model(
        sample_image.unsqueeze(0).to(train_module.engine.DEVICE)
    )

print(f"Fold 0 training samples: {len(train_loader.dataset):,}")
print(f"Fold 0 validation samples: {len(val_loader.dataset):,}")
print(f"Input shape: {tuple(sample_image.shape)}")
print(f"Output shape: {tuple(sample_output.shape)}")
print(f"Sample label: {int(sample_label)}")

del model
del train_loader
del val_loader
torch.cuda.empty_cache()

## 10. Mulai training 5-fold

Training dijalankan di kernel notebook, sehingga progress bar `tqdm` untuk train dan validation muncul langsung di bawah cell ini. Setiap fold memakai model baru. Model terbaik dipilih berdasarkan validation loss, sedangkan checkpoint terbaru disimpan pada setiap epoch.

In [ ]:
if DRIVE_OUTPUT_DIR.exists():
    raise FileExistsError(
        f"Output sudah ada: {DRIVE_OUTPUT_DIR}\n"
        "Ganti EXPERIMENT_ID untuk memulai training baru."
    )

print("Memulai baseline ResNet-50...", flush=True)
print(f"Input: {CT_PATH_COLUMN}", flush=True)
print(f"Fold: {NUM_FOLDS}", flush=True)
print(f"Epoch per fold: {NUM_EPOCHS}", flush=True)
print(f"Output: {DRIVE_OUTPUT_DIR}", flush=True)

train_module.main()

## 11. Lihat hasil cross-validation

Cell ini menampilkan ringkasan setiap fold, plot gabungan, dan lokasi artefak penting.

In [ ]:
import pandas as pd
from IPython.display import Image, display

summary_path = DRIVE_OUTPUT_DIR / "cv_summary.csv"
plot_path = DRIVE_OUTPUT_DIR / "figures/cv_fold_metrics.png"
config_snapshot_path = DRIVE_OUTPUT_DIR / "cv_resnet50.json"
oof_predictions_path = DRIVE_OUTPUT_DIR / "out_of_fold_predictions.csv"

if not summary_path.is_file():
    raise FileNotFoundError(f"Ringkasan training tidak ditemukan: {summary_path}")

display(pd.read_csv(summary_path))
if plot_path.is_file():
    display(Image(filename=str(plot_path), width=750))

print(f"Output directory: {DRIVE_OUTPUT_DIR}")
print(f"Config snapshot: {config_snapshot_path}")
print(f"OOF predictions: {oof_predictions_path}")
for fold in range(NUM_FOLDS):
    best_model_path = DRIVE_OUTPUT_DIR / f"fold_{fold}/best_model.pth"
    print(f"Fold {fold} best model: {best_model_path}")

## 12. Pengujian holdout dan XAI (opsional)

Ubah `RUN_TEST_AFTER_TRAINING=True` pada cell konfigurasi untuk mengevaluasi ensemble lima model pada holdout test serta menghasilkan Grad-CAM dan LRP. Proses XAI untuk seluruh holdout dapat memerlukan waktu lama; gunakan `MAX_TEST_SAMPLES=8` untuk smoke test. Progress `tqdm` juga ditampilkan oleh script pengujian.

In [ ]:
if RUN_TEST_AFTER_TRAINING:
    test_command = [
        sys.executable,
        "-m",
        "003_classification.cv_resnet50.test",
        str(DRIVE_OUTPUT_DIR),
        "--batch-size",
        str(TEST_BATCH_SIZE),
        "--num-workers",
        str(TEST_NUM_WORKERS),
        "--device",
        DEVICE,
        "--dpi",
        "120",
    ]

    if MAX_TEST_SAMPLES is not None:
        test_command.extend(["--max-samples", str(MAX_TEST_SAMPLES)])

    subprocess.run(test_command, cwd=PROJECT_ROOT, check=True)
    print(f"Hasil test: {DRIVE_OUTPUT_DIR / 'test'}")
else:
    print("Pengujian dilewati karena RUN_TEST_AFTER_TRAINING=False.")

## Menjalankan eksperimen baru

Engine CV saat ini membuat satu direktori baru dan tidak melanjutkan fold yang terputus secara otomatis. Untuk eksperimen baru, ubah `EXPERIMENT_ID`, sesuaikan konfigurasi, lalu jalankan kembali mulai dari cell konfigurasi. Jangan hapus hasil lama di Google Drive sebelum memastikan semua artefak yang dibutuhkan sudah tersimpan.